# RAGcore


## 1. .env settings 

In [39]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
load_dotenv(Path("../.env"))


http_proxy = os.getenv("http_proxy")
https_proxy = os.getenv("https_proxy")
HTTP_PROXY = os.getenv("HTTP_PROXY")
HTTPS_PROXY = os.getenv("HTTPS_PROXY")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")

assert GROQ_API_KEY, "GROQ_API_KEY not found — check that .env exists in the repo root and has a real key."
print("Setup OK")
print(f"  GROQ_MODEL = {GROQ_MODEL}")
print(f"  EMBEDDING_MODEL = {EMBEDDING_MODEL}")

Setup OK
  GROQ_MODEL = llama-3.3-70b-versatile
  EMBEDDING_MODEL = sentence-transformers/all-MiniLM-L6-v2


## 2. Load a document

Extract text from a PDF, page by page

In [77]:
from pypdf import PdfReader

from pathlib import Path

SAMPLE_PDF_PATH = str(Path("../data/uploads/Deep Learning by Ian Goodfellow, Yoshua Bengio, Aaron Courville.pdf"))

reader = PdfReader(SAMPLE_PDF_PATH)
print(f"Loaded {len(reader.pages)} page(s) from {SAMPLE_PDF_PATH}")

pages_text = []
for page_num, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    if text.strip():
        pages_text.append({"page": page_num, "text": text})

print(f"Extracted text from {len(pages_text)} non-empty page(s)")
print("\n--- Preview of page 1 ---")
print(pages_text[0]["text"][:500])

Loaded 801 page(s) from ../data/uploads/Deep Learning by Ian Goodfellow, Yoshua Bengio, Aaron Courville.pdf
Extracted text from 800 non-empty page(s)

--- Preview of page 1 ---
Deep Learning
Ian Goodfellow
Yoshua Bengio
Aaron Courville


## 3. Chunking

In [78]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", 1000))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", 150))

page_documents = [
    Document(page_content=p["text"], metadata={"source": SAMPLE_PDF_PATH, "page": p["page"]})
    for p in pages_text
]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(page_documents)

for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_index"] = i

print(f"Split {len(page_documents)} page(s) into {len(chunks)} chunk(s)\n")
print(f" No of chunks: {len(chunks)},\n chunk_size={CHUNK_SIZE},\n chunk_overlap={CHUNK_OVERLAP}")
print("\n--- Preview of chunk 0 ---")
print(chunks[0].page_content[:300])
print("\nMetadata:", chunks[0].metadata)

Split 800 page(s) into 2322 chunk(s)

 No of chunks: 2322,
 chunk_size=1000,
 chunk_overlap=150

--- Preview of chunk 0 ---
Deep Learning
Ian Goodfellow
Yoshua Bengio
Aaron Courville

Metadata: {'source': '../data/uploads/Deep Learning by Ian Goodfellow, Yoshua Bengio, Aaron Courville.pdf', 'page': 2, 'chunk_index': 0}


## 4. Embedding

In [80]:
import os
from langchain_huggingface import HuggingFaceEmbeddings

# HF_HOME default — avoids downloading the model a second time if you've already used it through the running application.
HF_HOME = Path("../.hf_cache").resolve()
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},  # keeps similarity scores well-behaved
)

# Embed just one chunk to see what an embedding actually is.
sample_vector = embedding_model.embed_query(chunks[0].page_content)

print(f"Using HF_HOME")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Vector dimension: {len(sample_vector)}")
print(f"First 8 values: {sample_vector[:8]}")

Using HF_HOME
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Vector dimension: 384
First 8 values: [-0.06303867697715759, -0.016751857474446297, 0.02572939544916153, -0.030512986704707146, -0.08692377060651779, 0.0557619072496891, 0.009984896518290043, -0.06722758710384369]


### what's actually happening under the hood

`HuggingFaceEmbeddings` (via `sentence-transformers`) is already running on PyTorch — it's a convenience wrapper around exactly these steps: tokenize → forward pass through the transformer → pool token vectors into one sentence vector → normalize. Let's do it manually, with raw `transformers` + `torch`, to see the mechanics directly and confirm we get the same result.

In [81]:
import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F


raw_tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
raw_model = AutoModel.from_pretrained(EMBEDDING_MODEL)
raw_model.eval()  # inference mode — disables dropout etc.


def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]  # shape: (batch, seq_len, hidden_dim)
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    summed = torch.sum(token_embeddings * mask, dim=1)
    counted = torch.clamp(mask.sum(dim=1), min=1e-9)  # avoid divide-by-zero
    return summed / counted


def embed_raw_pytorch(text: str) -> torch.Tensor:
    tokens = raw_tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():  # no gradient tracking needed — we're not training
        output = raw_model(**tokens)
    embedding = mean_pooling(output, tokens["attention_mask"])
    embedding = F.normalize(embedding, p=2, dim=1)  # same normalize_embeddings=True as before
    return embedding.squeeze()


# Compare against the sentence-transformers wrapper's output for the same text
sample_text = chunks[100].page_content
raw_vector = embed_raw_pytorch(sample_text)
wrapped_vector = embedding_model.embed_query(sample_text)

print(f"Raw PyTorch vector shape: {tuple(raw_vector.shape)}")
print(f"Raw PyTorch first 8 values: {raw_vector[:8].tolist()}")
print(f"\nHuggingFaceEmbeddings first 8 values: {wrapped_vector[:8]}")

similarity = F.cosine_similarity(
    raw_vector.unsqueeze(0),
    torch.tensor(wrapped_vector).unsqueeze(0)
)
print(f"\nCosine similarity between raw-PyTorch and wrapped output: {similarity.item():.6f}")

Raw PyTorch vector shape: (384,)
Raw PyTorch first 8 values: [-0.08621171861886978, -0.0013607284054160118, 0.07335063070058823, -0.019600585103034973, 0.04947417601943016, 0.01534206047654152, 0.008418316021561623, -0.003463044064119458]

HuggingFaceEmbeddings first 8 values: [-0.08621171861886978, -0.0013607284054160118, 0.07335063070058823, -0.019600585103034973, 0.04947417601943016, 0.01534206047654152, 0.008418316021561623, -0.003463044064119458]

Cosine similarity between raw-PyTorch and wrapped output: 1.000000


## 5. Vector store (Chroma)

Embed all the chunks and store them in a persistent Chroma collection — this is what makes similarity search possible: instead of comparing text directly, we compare the embedding vectors. Uses a separate collection (`notebook_demo`) so this doesn't touch the web app's actual data.

In [86]:
from langchain_chroma import Chroma

NOTEBOOK_CHROMA_DIR = str(Path("./chroma_demo").resolve())

vectorstore = Chroma(
    collection_name="notebook_demo",
    embedding_function=embedding_model,
    persist_directory=NOTEBOOK_CHROMA_DIR,
)

# Deterministic IDs (chunk_index-based) mean re-running this cell
# overwrites the same vectors instead of duplicating them.
ids = [f"chunk_{c.metadata['chunk_index']}" for c in chunks]

vectorstore.add_documents(documents=chunks, ids=ids)

print(f"Stored {len(chunks)} chunk(s) in Chroma collection 'notebook_demo'")
print(f"Persisted to Directory")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Stored 2322 chunk(s) in Chroma collection 'notebook_demo'
Persisted to Directory
